In [ ]:
# Mission 3: 로컬 기본 실행, Colab에서는 기존 Drive 경로 사용


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

current = Path.cwd().resolve()
REPO_ROOT = next((p for p in (current, *current.parents) if (p / "src/dcc_data").is_dir() and (p / "scripts").is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError(f"저장소를 준비한 뒤 그 안에서 실행하세요: {current}")
PYTHON_EXECUTABLE = os.environ.get("DCC_PYTHON", sys.executable)
if not Path(PYTHON_EXECUTABLE).is_file():
    raise FileNotFoundError(f"Python executable not found: {PYTHON_EXECUTABLE}")
try:
    from google.colab import drive as colab_drive
except ImportError:
    colab_drive = None
IN_COLAB = colab_drive is not None
if IN_COLAB and "DCC_DATA_ROOT" not in os.environ:
    colab_drive.mount("/content/drive")
if str(REPO_ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "scripts"))


In [ ]:
default_data = Path("/content/drive/MyDrive/대학부 데이터") if IN_COLAB else REPO_ROOT / "data/raw_original"
DATA_DIR = Path(os.environ.get("DCC_DATA_ROOT", default_data)).expanduser().resolve()
TRAIN_LABEL_DIR = DATA_DIR / "Training/2.라벨링데이터/TL_서울_구급"
VALID_LABEL_DIR = DATA_DIR / "Validation/2.라벨링데이터/VL_서울_구급"
VALID_AUDIO_DIR = DATA_DIR / "Validation/1.원천데이터/VS_서울_구급"
for folder in (TRAIN_LABEL_DIR, VALID_LABEL_DIR, VALID_AUDIO_DIR):
    if not folder.is_dir():
        raise FileNotFoundError(f"공식 데이터 폴더를 찾지 못했습니다: {folder}")
ARTIFACT_DIR = REPO_ROOT / "artifacts"
PREDICTION_PATH = REPO_ROOT / "outputs/mission3.csv"
print("Repository:", REPO_ROOT)
print("Data:", DATA_DIR)


In [ ]:
train_json_files = sorted(TRAIN_LABEL_DIR.glob("*.json"))
print("Training JSON 개수:", len(train_json_files))
for json_file in train_json_files[:5]:
    print(json_file.name)
assert train_json_files, f"Training JSON을 찾지 못했습니다: {TRAIN_LABEL_DIR}"


In [ ]:
# JSON 파일을 읽기 위한 도구입니다.
import json


# 첫 번째 Training JSON 파일을 엽니다.
with train_json_files[0].open(
    "r",
    encoding="utf-8-sig",
) as file:

    # JSON 내용을 파이썬 딕셔너리로 변환합니다.
    sample_data = json.load(file)


# JSON에 어떤 항목들이 들어 있는지 확인합니다.
print("전체 JSON 항목:")
print(sample_data.keys())


# 같은 통화를 구분하는 번호를 확인합니다.
print(
    "\nrecordId:",
    sample_data.get("recordId"),
)


# 환자의 증상 정답을 확인합니다.
print(
    "\nsymptom:",
    sample_data.get("symptom"),
)


# 대화가 몇 문장으로 구성되어 있는지 확인합니다.
print(
    "\n발화 개수:",
    len(sample_data.get("utterances", [])),
)


# 첫 번째 발화 내용을 확인합니다.
print(
    "\n첫 번째 발화:",
    sample_data.get("utterances", [])[0],
)

In [ ]:
from mission3_baseline import load_training_dataframe

dataframe = load_training_dataframe(TRAIN_LABEL_DIR)
print("불러온 데이터 개수:", len(dataframe))
display(dataframe.head())


In [ ]:
# 평가 대상 증상과 평가 대상에서 제외된 증상을 구분해 표시한다.
display(dataframe[["label_file_name", "symptoms", "non_target_symptoms"]].head(10))


In [ ]:
# 이 셀을 실행할 때만 학습한다. Python과 실행 파일 위치를 명시한다.
subprocess.run([
    PYTHON_EXECUTABLE, str(REPO_ROOT / "scripts/mission3_baseline.py"),
    "--label_dir", str(TRAIN_LABEL_DIR), "--output_dir", str(ARTIFACT_DIR),
    "--threshold", "0.5", "--test_size", "0.2", "--seed", "42",
], cwd=REPO_ROOT, check=True)


In [ ]:
# 학습 직후 저장된 baseline 평가 결과를 확인합니다.\nimport json\nimport pandas as pd\n\nwith (ARTIFACT_DIR / "metrics.json").open("r", encoding="utf-8") as file:\n    baseline_metrics = json.load(file)\n\nprint("===== Baseline 평가 결과 =====")\nprint(f"Macro F1        : {baseline_metrics['macro_f1']:.4f}")\nprint(f"Macro Precision : {baseline_metrics['macro_precision']:.4f}")\nprint(f"Macro Recall    : {baseline_metrics['macro_recall']:.4f}")\nprint(f"Micro F1        : {baseline_metrics['micro_f1']:.4f}")\nprint(f"Subset Accuracy : {baseline_metrics['subset_accuracy']:.4f}")\nprint(f"최저 F1 증상     : {baseline_metrics['worst_symptom']} ({baseline_metrics['worst_symptom_f1']:.4f})")\nprint(f"동일 원본 행 중복 : {baseline_metrics['train_validation_index_overlap_count']}")\nprint(f"동일 전사 중복    : {baseline_metrics['train_validation_duplicate_text_count']}")\n\nper_label_metrics = pd.read_csv(ARTIFACT_DIR / "per_label_metrics.csv")\ndisplay(per_label_metrics.sort_values("f1", ascending=True).reset_index(drop=True))\n

In [ ]:
# 가장 F1이 낮은 증상의 FN/FP 예시를 사람이 직접 읽어봅니다.\n# FN = 실제 증상이 있는데 모델이 놓친 경우\n# FP = 실제 증상이 없는데 모델이 있다고 판단한 경우\nerror_examples = pd.read_csv(ARTIFACT_DIR / "worst_label_error_examples.csv")\n\nprint("===== 최저 F1 증상 오류 사례 =====")\nif error_examples.empty:\n    print("저장된 FN/FP 사례가 없습니다.")\nelse:\n    display(error_examples)\n

In [ ]:
# 이 셀을 실행할 때만 공식 Validation 추론을 수행한다.
subprocess.run([
    PYTHON_EXECUTABLE, str(REPO_ROOT / "scripts/inference.py"),
    "--audio_dir", str(VALID_AUDIO_DIR), "--label_dir", str(VALID_LABEL_DIR),
    "--ckpt_path", str(ARTIFACT_DIR / "mission3_tfidf_logreg.joblib"),
    "--output", str(PREDICTION_PATH),
], cwd=REPO_ROOT, check=True)
